<a href="https://colab.research.google.com/github/nehnamehranmk638-dev/multilingual-rag-research/blob/nehna/notebooks/ml_06_faithfulness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Malayalam Phase 1 — Step 6: Faithfulness Evaluation

Mirrors English `06_faithfulness.ipynb`.

**What this notebook does:**
1. Samples 30 questions for manual review
2. Displays each question + retrieved passages + model answer for you to label
3. Records your label: `supported`, `not_supported`, or `partial`
4. Records error type: `retrieval_missed`, `generator_ignored_context`, or `none`
5. Saves `faithfulness_labels_ml.json`

> **Run sections 1–3 first (setup + data load), then label one at a time in section 4.**

## 1. GitHub setup

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [ ]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git

fatal: destination path 'multilingual-rag-research' already exists and is not an empty directory.


In [ ]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [ ]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")

GitHub authentication configured.


In [ ]:
!git fetch origin
!git switch nehna
!git pull

Already on 'nehna'
Your branch is up to date with 'origin/nehna'.
Already up to date.


## 2. Load data

In [ ]:
import json
import random

with open("data/questions_ml.json", "r", encoding="utf-8") as f:
    questions_ml = json.load(f)

with open("data/corpus_ml.json", "r", encoding="utf-8") as f:
    corpus_ml = json.load(f)

with open("results/hybrid_top10_ml.json", "r", encoding="utf-8") as f:
    hybrid_results_ml = json.load(f)

with open("results/predictions_hybrid_ml.json", "r", encoding="utf-8") as f:
    predictions_ml = json.load(f)

print("Questions    :", len(questions_ml))
print("Corpus       :", len(corpus_ml))
print("Hybrid IDs   :", len(hybrid_results_ml))
print("Predictions  :", len(predictions_ml))

Questions    : 100
Corpus       : 247
Hybrid IDs   : 100
Predictions  : 100


## 3. Sample 30 questions for manual review

Same seed (7) as English notebook for reproducibility.

In [ ]:
random.seed(7)
sample_questions_ml = random.sample(questions_ml, 30)

print("Manual sample size:", len(sample_questions_ml))

Manual sample size: 30


In [ ]:
def show_case_ml(q):
    """Display one case for manual labeling."""
    qid = q["question_id"]
    top_ids = hybrid_results_ml[str(qid)][:5]
    passages = [corpus_ml[pid]["text"] for pid in top_ids]

    print("=" * 80)
    print("QUESTION    :", q["question"])
    print("GOLD ANSWER :", q["gold_answers"])
    print("MODEL ANSWER:", predictions_ml[str(qid)])

    # Check if gold answer appears in any retrieved passage
    gold_in_retrieved = any(
        any(g in corpus_ml[pid]["text"] for g in q["gold_answers"])
        for pid in top_ids
    )
    print("Gold in retrieved passages:", gold_in_retrieved)

    print("\nRETRIEVED PASSAGES:")
    for i, p in enumerate(passages, start=1):
        print(f"[{i}] {p[:400]}")
        print()
    print("=" * 80)

## 4. Manual labeling

**Label guide:**
- `supported` — model answer is correct and supported by retrieved passages
- `not_supported` — model answer is wrong or not in retrieved passages
- `partial` — answer is partly correct

**Error types (only for `not_supported`):**
- `retrieval_missed` — correct passage was NOT in top-5 retrieved
- `generator_ignored_context` — correct passage WAS retrieved but model ignored it
- `none` — for `supported` or `partial`

Run `show_case_ml(sample_questions_ml[N])` to see case N, then run the `labels_ml.append(...)` cell below it.

In [ ]:
# Initialize labels list — run this ONCE at the start
labels_ml = []

In [ ]:
show_case_ml(sample_questions_ml[0])

QUESTION    : ഇന്ത്യയിലെ  4 പ്രധാന പർവതനിരകൾ ഏതെല്ലാം ?
GOLD ANSWER : ['ആരവല്ലി, സത്പുര, വിന്ധ്യൻ, പശ്ചിമഘട്ടം']
MODEL ANSWER: ൂമിയിൽ പ്രായിക്കുന്ന പ
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] പൂർണ്ണമായും ഭൂമിയുടെ ഉത്തരാർദ്ധ ഗോളത്തിലാണ്‌ ഇന്ത്യയുടെ സ്ഥാനം. ഉത്തര അക്ഷാംശം 8 ഡിഗ്രി 7 മിനുറ്റിനും 37 ഡിഗ്രി 6 മിനുറ്റിനും മദ്ധ്യേയും പൂർവ രേഖാംശം 68 ഡിഗ്രി  മിനുറ്റിനും 97 ഡിഗ്രി 25 മിനുറ്റിനും മദ്ധ്യേയുമാണ്‌ ഇന്ത്യയുടെ സ്ഥാനം. ലോകത്തിലെ ഭൂവിസ്തൃതിയുടെ 2.42 ശതമാനവും എന്നാൽ ജനസംഖ്യയുടെ 16 ശതമാനവും ഇന്ത്യയിലുണ്ട്. വലിപ്പത്തിൽ ലോകരാഷ്ട്രങ്ങൾക്കിടയിൽ 7 ആം സ്ഥാനവും ജനസംഖ്യയിൽ രണ്ടാംസ്ഥാനവും ഇന്ത്യക

[2] ഇന്ത്യയിലെ ഏറ്റവുമധികം ചരിത്രപ്രാധാന്യമുള്ള നഗരങ്ങളിലൊന്നാണ്‌ ദില്ലി. 7 നഗരങ്ങളുടേയും ആയിരം സ്മാരകങ്ങളുടേയും നഗരം എന്നാണ് ദില്ലിയെപ്പറ്റി പരാമർശിക്കുന്നത്. പതിനൊന്ന് പ്രധാനപ്പെട്ട ചക്രവർത്തിമാരുടെ ശവകുടിരങ്ങൾ ദില്ലിയിലുണ്ട്. ഇന്ത്യയിലെ മറ്റുപ്രദേശങ്ങളിലൊട്ടാകെ നോക്കിയാലും ഇത്തരത്തിലുള്ള നാലെണ്ണം മാത്രമേയുള്ളൂ‌. ക്രിസ്തുവർഷത്തിന്റെ ആരംഭത്തിൽത്തന്നെ ദില്ലിയിലെ ആദ്യനഗരമായ ഇന്ദ്രപ്രസ്ഥം 

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[0]["question_id"],
    "faithfulness": "not_supported",       # CHANGE: supported / not_supported / partial
    "error_type": "generator_ignored_context"                  # CHANGE: None / retrieval_missed / generator_ignored_context / none
})

In [ ]:
show_case_ml(sample_questions_ml[1])

QUESTION    : ഇന്ത്യയിലെ രണ്ടാമത്തെ ഭൂഗർഭ മെട്രോ ശൃംഖല ഏതാണ്?
GOLD ANSWER : ['ഡെൽഹി മെട്രോ']
MODEL ANSWER: �ടക്കമായ കൊല്ലം ആയിരുന്�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] രാജ്യത്തിന്റെ എല്ലാ ഭാഗത്തേക്കുമുള്ള ദീർഘദൂര ട്രെയിനുകൾ ഇവിടെ നിന്നും പുറപ്പെടുന്നു. ഓൾഡ് ഡെൽഹിയിലുള്ള പുരാണ ദില്ലി സ്റ്റേഷനും വളരെയധികം യാത്രക്കാരുള്ള ഒരു സ്റ്റേഷനാണ്‌. നഗരത്തിന്റെ പ്രാന്തപ്രദേശങ്ങളിലേക്കുള്ള സബർബൻ റെയിൽ‌വേ സർ‌വീസുകളും ഇവിടെ നിന്നുണ്ട്. ന്യൂ ഡെൽഹി നഗരത്തേയും പരിസരപ്രദേശങ്ങളേയും ബന്ധിപ്പിക്കുന്ന അതിവേഗ മെട്രോറെയിൽ സർ‌വീസ് 2004 ഡിസംബർ 24-നാണ്‌ പ്രവർത്തനമാരംഭിച്ചത്. ഇന്ത്യയിലെ രണ്ടാ

[2] താജ് മഹൽ പണിതീർത്തിരിക്കുന്നത് ചുവരുകളുടെ നഗരം എന്നറിയപ്പെടുന്ന  ആഗ്ര നഗരത്തിന്റെ തെക്ക് ഭാഗത്ത് യമുന നദിയുടെ തീരത്താണ്. മഹാരാജ ജയ് സിംഗിൽ  നിന്നും വാങ്ങിയ ഭൂമിയായിരുന്നു ഇത്. പകരമായി ഷാജഹാൻ മഹാരാജ ജയ് സിങിന് ഒരു കൊട്ടാരം നൽകി എന്നാണ് പറയപ്പെടുന്നത്. മൂന്ന് ഏക്കറോളം വരുന്ന ഭൂമി ആദ്യം നിരപ്പാക്കി എടുക്കുകയും പിന്നീട് യമുന നദിയുടെ നിരപ്പിൽ നിന്നും 50 മീറ്ററോളം ഉയരത്തിൽ നിരത്തി എടുക്കുകയുമായിരുന്നു. 

[3

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[1]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[2])

QUESTION    : സുംഗ രാജവംശത്തിനുശേഷം, ഏത്  രാജവംശമാണ് മഗധയിൽ അധികാരത്തിലെത്തിയത് ?
GOLD ANSWER : ['കണ്വവംശം']
MODEL ANSWER: യി നിന്നുള്ള പുതിയ തി
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] പ്രാക്കാലത്ത് മധ്യദേശം എന്നറിയപ്പെട്ടിരുന്ന മേഖലയാണ് ഇന്നത്തെ ഉത്തർപ്രദേശ്. തന്ത്ര പ്രധാനമായ സ്ഥാനം‌‌മൂലം ഉത്തരേന്ത്യയുടെ ചരിത്രവുമായി ഈ പ്രദേശത്തിന് അഭേദ്യമായ ബന്ധമുണ്ട്. മിർസാപൂർ, ബുന്ദേൽഖണ്ഡ് എന്നിവിടങ്ങളിൽനിന്നു കണ്ടെടുത്ത ചരിത്രാവശിഷ്ടങ്ങൾ ഉത്തർപ്രദേശിന്റെ പൗരാണികത്വത്തിലേക്കു വെളിച്ചം വീശുന്നു. ഋഗ്‌‌വേദകാലം മുതൽ മാത്രമേ ഏതാണ്ട് സുവ്യക്തമായ ഒരു ചരിത്രം ഈ പ്രദേശത്തെ സംബന്ധിച്ച് ലഭിക്കുന്നുള്ളു

[2] ലോകത്തിലെ ഏറ്റവും പ്രശസ്തമായ ജനാധിപത്യ വ്യവസ്ഥിതി നിലനിൽക്കുന്നത് ഇന്ത്യയിലാണ്‌. സ്വാതന്ത്ര്യം കിട്ടിയതിനു ശേഷം മിക്കവാറും വർഷങ്ങൾ ഇന്ത്യ ഭരിച്ചത് ഇന്ത്യൻ നാഷണൽ കോൺഗ്രസ് നയിച്ച ഗവൺ‌മെന്റ് ആയിരുന്നു ഇന്ത്യൻ സംസ്ഥാനങ്ങളിലെ ദേശീയ പാർട്ടികളിൽ ഇന്ത്യൻ നാഷണൽ കോൺഗ്രസ്, ഭാരതീയ ജനതാ പാർട്ടി) എന്നീ ദേശീയ പാർട്ടികളും മറ്റനേകം പ്രാദേശിക പാർട്ടികളും ഉൾപ്പെടുന്നു. 1950-നും 1990-നും ഇടയിൽ രണ്ടു തവണയൊഴിക

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[2]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[3])

QUESTION    : രാഷ്ട്ര പാതി ഭവനത്തിന്റെ മാറ്റാനുള്ള ലുറ്റിയൻസിന്റെ നിർദ്ദേശം നിരസിച്ചത് ഏത് കമ്മിറ്റി ആയിരുന്നു   ?
GOLD ANSWER : ['ഇംപീരിയൽ ഡെൽഹി കമ്മി']
MODEL ANSWER: ്തിന്റെ ദൃഷ്ടിയും പേര
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] പൂർത്തിയാക്കിയ ശേഷം, ലുട്ടെൻസ് ബേക്കറുമായി തർക്കിച്ചു, കാരണം കെട്ടിടത്തിന്റെ മുൻവശത്തെ കാഴ്ച റോഡിന്റെ ഉയർന്ന കോണിൽ അവ്യക്തമായിരുന്നു. ല്യൂട്ടിയൻസ് ഇത് പരിഹരിക്കുന്നതിനായി ശ്രെമം നടത്തിയെങ്കിലും വിജയിച്ചില്ല. ഇരുവശത്തും ഭിത്തികൾ നിലനിർത്തിക്കൊണ്ട് വൈസ്രോയിയുടെ വീട്ടിലേക്ക് ഒരു നീണ്ട ചെരിഞ്ഞ ഗ്രേഡ് ഉണ്ടാക്കാൻ ല്യൂട്ടിയൻസ് ആഗ്രഹിച്ചു. ഇത് വീടിന്റെ പുറകിൽ നിന്ന് ഒരു കാഴ്ച നൽകുമെങ്കിലും, സെക്രട്ടേറിയറ്

[2] കലാപസമയത്ത് കൊട്ടാരത്തിൽ അഭയം തേടിയിരുന്ന ചില ഇംഗ്ലീഷുകാർ റാണിക്ക് ഈ കലാപകാരികളുമായി യാതൊരു തരത്തിലും ബന്ധമുണ്ടായിരുന്നില്ലെന്ന് തെളിവുകളെ ഉദ്ധരിച്ചുകൊണ്ട് സമർത്ഥിക്കുന്നു. ഡെൽഹിയിലേക്കു പോകാനായി തങ്ങൾക്ക് മൂന്നു ലക്ഷം രൂപ നൽകണമെന്നും അതിനു തയ്യാറായില്ലെങ്കിൽ കൊട്ടാരം തീവെച്ചു നശിപ്പിക്കുമെന്നും കോട്ട വളഞ്ഞുകൊണ്ട് ശിപായിമാർ റാണിയോട് ആവശ്യപ്പെട്

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[3]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[4])

QUESTION    : നവോത്ഥാന നായകന്മാരായ ശ്രീനാരായണ ഗുരുവിന്റെയും അയ്യങ്കാളിയുടെയും കേന്ദ്രം  ഏതായിരുന്നു ?
GOLD ANSWER : ['കൊല്ലവും']
MODEL ANSWER: റോഡാ രാജാവായിരുന്ന �
Gold in retrieved passages: False

RETRIEVED PASSAGES:
[1] ഗുഹാനന്ദപുരം സുബ്രഹ്മണ്യസ്വാമി ക്ഷേത്രം,കൊല്ലംജില്ലയിലെതെക്കുംഭാഗംഗ്രാമത്തിൽ സ്ഥിതി ചെയ്യുന്നു. സ്ഥാപിച്ചത് ശ്രീനാരായണ ഗുരു ശിഷ്യനായബ്രഹ്മശ്രീ ഷൺമുഖദാസ് സ്വാമികൾ ഓച്ചിറ പരബ്രഹ്മക്ഷേത്രം ,പത്തനംതിട്ടജില്ല ഉണ്ടാകുന്നതിനു മുൻപ് പ്രസിദ്ധ ഹിന്ദു ക്ഷേത്രമായ ശബരിമല ക്ഷേത്രം, കൊല്ലംജില്ലയിൽ ആയിരുന്നു. ആശ്രാമം ശ്രീകൃഷ്ണക്ഷേത്രം, പുതിയകാവ് ഭഗവതി ക്ഷേത്രം കൊല്ലം റെയിൽവേ സ്റ്റേഷനടുത്തായി സ്ഥിതി ചെയ്യുന്നു. തൃ

[2] ഇന്ത്യയിലേക്ക് മടങ്ങിയ ഗാന്ധി 1915 ജനുവരി 9 ന് മുംബൈ തുറമുഖത്ത് കപ്പലിറങ്ങി (ഈ ദിനത്തിന്റെ ഓർമക്കായിട്ടാണ് 2003 മുതൽ ജനുവരി 9 ഭാരത സർക്കാർ പ്രവാസി ദിനമായി ആചരിക്കാൻ തുടങ്ങിയത്). കാര്യങ്ങൾ നേരിട്ട് പഠിക്കാനായി അദ്ദേഹം ഇന്ത്യ മുഴുവനും സഞ്ചരിച്ചു. അന്ന് ഇൻഡ്യൻ ദേശീയതയുടെ നായകരായിരുന്ന നേതാക്കന്മാരെയെല്ലാം സന്ദർശിച്ചു. ദക്ഷിണാഫ്രിക്കയിലെ ഫീനിക്സ് ആശ്രമത്തിൽ നിന്നും വന്ന വി

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[4]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[5])

QUESTION    : സി വി രാമന്റെ  ജന്മസ്ഥലം ഏത്  ?
GOLD ANSWER : ['തമിഴ്\u200cനാട്ടിലെ തഞ്ചാവൂർ ജില്ലയിൽ']
MODEL ANSWER: ിർബന്ധമായി കൈക്കുകയും
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] 1888 നവംബർ 7-ന്, തമിഴ്‌നാട്ടിലെ തഞ്ചാവൂർ ജില്ലയിൽ,  ചന്ദ്രശേഖര അയ്യരുടേയും പാർവതി അമ്മാളുടേയും രണ്ടാമത്തെ മകനായി ചന്ദ്രശേഖര വെങ്കിട്ടരാമൻ ജനിച്ചു. അഞ്ച് ആണ്മക്കളും മൂന്നു പെണ്മക്കളുമായി ഈ ദമ്പതികൾക്ക് എട്ട് മക്കളാണ് ഉണ്ടായിരുന്നത്. ബ്രാഹ്മണ കുടുംബത്തിലാണ് ജനിച്ചതെങ്കിലും രാമന് ജാതി മത ചടങ്ങുകളിൽ അത്ര താത്പര്യമില്ലായിരുന്നു. മന് നാലുവയസ്സുള്ളപ്പോൾ, പിതാവിന്  വിശാഖപട്ടണത്തുള്ള കോളേജിൽ  അദ്ധ്യാപകനായി

[2] ഇതിനെ തടഞ്ഞ ജടായുവിനെ രാവണൻ വധിച്ചു. സീതയെ അന്വേഷിച്ചു നടക്കുന്ന രാമലക്ഷ്മണന്മാർ പമ്പാ തീരത്തുവച്ച് ഹനുമാനെ കാണുന്നു. ഹനുമാന്റെ ആഗ്രഹപ്രകാരം സുഗ്രീവനേയും അവർ പരിചയപ്പെടുന്നു. സുഗ്രീവനെ സ്വന്തം ജ്യേഷ്ഠനായ ബാലിയുടെ നിയന്ത്രണങ്ങളിൽ നിന്ന് മോചിപ്പിച്ച് ബാലിവധം നടത്തുന്നു. പകരമായി സീതയെ അന്വേഷിക്കുന്ന ചുമതല സുഗ്രീവൻ തന്റെ വാനരസേനയെ ഏല്പിക്കുന്നു. വാനരപ്പടയിൽ പക്ഷിശ്രേഷ്ഠനും ജടായുവിന്റെ സഹോദരനുമായ സമ്

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[5]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[6])

QUESTION    : ആർക്കുള്ള  മറുപടിയായിട്ടാണ്  ഞാൻ എന്തുകൊണ്ടാണ് ഒരു നിരീശ്വരവാദിയായത് എന്നതിനെക്കുറിച്ച് ഭഗത് ഒരു ബുക്ക്‌ലെറ്റ് എഴുതിയത്?
GOLD ANSWER : ['രൺധീറിനോടുള്ള മറുപടിയായാണ്']
MODEL ANSWER: ുടങ്ങിയ സാധ്യതകൾ പ്രസിദ്�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ഭഗത് സിംഗ് ജയിലിലായിരുന്നപ്പോൾ അദ്ദേഹത്തെ വധശിക്ഷയിൽ നിന്നും രക്ഷിക്കാമായിരുന്നിട്ടും, മഹാത്മാഗാന്ധി അത് ചെയ്തില്ല എന്നു പറയപ്പെടുന്നു. ഭഗത് സിംഗിന്റെ വധശിക്ഷ പെട്ടെന്ന് നടപ്പിലാക്കാൻ മോഹൻ ദാസ് കരംചന്ദ് ഗാന്ധി ബ്രിട്ടീഷുകാർക്കൊപ്പം ഗൂഢാലോചന നടത്തിയെന്നു വരെ പറയപ്പെടുന്നു. ഭഗത് സിംഗിന്റെ വധശിക്ഷ റദ്ദാക്കാൻ ബ്രിട്ടനുമേൽ സമ്മർദ്ദം ചെലുത്താനുള്ള സ്വാധീനം ഗാന്ധിജിക്കുണ്ടായിരുന്നില്ല എന്നു പറഞ്ഞ് അദ്ദേഹ

[2] 1910 ന്റെ തുടക്കത്തിൽ അദ്ദേഹം ടോക്കിയോയിൽ ഇസ്ലാമിക സാഹോദര്യം എന്ന പേരിൽ സംഘടന ആരംഭിച്ചു. 1911 ജൂൺ-ജൂലൈ മാസങ്ങളിൽ അദ്ദേഹം കോൺസ്റ്റാന്റിനോപ്പിളിലേക്കും പെട്രോഗ്രാഡിലേക്കും പോയി, ഒക്ടോബറിൽ ടോക്കിയോയിലേക്ക് മടങ്ങിയ അദ്ദേഹം, അഫ്ഗാനിസ്ഥാൻ ഉൾപ്പെടെയുള്ള ഒരു വലിയ പാൻ-ഇസ്ലാമിക് സഖ്യത്തിന്റെ വരവിനെ പരാമർശിക്കുന്ന "മധ്യേഷ്യയു

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[6]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[7])

QUESTION    : 1150 മുതൽ 1280 CE വരെ, ചോളരുടെ ശക്തരായ എതിരാളികൾ ആരായിരുന്നു?
GOLD ANSWER : ['പാണ്ഡ്യ രാജാക്കന്മാരായിരുന്നു']
MODEL ANSWER: യ അന്തരിഫലിക്കുകളിലുള്ള �
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] കലചൂരികൾ ചാലൂക്യ തലസ്ഥാനം ഏകദേശം 1135 മുതൽ 35 വർഷത്തേയ്ക്ക് കീഴടക്കിവെച്ചു. വടക്കൻ കർണ്ണാ‍ടകത്തിലെ ധാർവാഡ് വിഷ്ണുവർദ്ധനന്റെ കീഴിൽ ഹൊയ്സാലർ കീഴടക്കി, ഇവിടെ വിഷ്ണുവർദ്ധനൻ ഭരിക്കുകയും, ഹൊയ്സാ‍ല തലസ്ഥാനമായ ദ്വാരസമുദ്രം തന്റെ മകനായ നരസിംഹൻ ഒന്നാമനെ ഏകദേശം ക്രി. വ. 1149-ൽ ഏൽ‌പ്പിക്കുകയും ചെയ്തു. ഈ സമയത്തു തന്നെ, ക്രി. വ. 1120-നു ശേഷം ചാലൂക്യ ഭരണാ‍ധികാരികളുടെ പിടിപ്പുകേടുകൊണ്ട്, ചാലൂക്യ സാമ്രാജ്യം ക്ഷയിക

[2] ചോളരുടെ ചരിത്രത്തെ നാലു കാലഘട്ടങ്ങളായി വിഭജിക്കാം: സംഘസാഹിത്യ കാലത്തെ ആദ്യകാല ചോളർ, സംഘം ചോളരുടെ പതനത്തിനും വിജയലായന്റെ കീഴിൽ (ക്രി. വ. 848) മദ്ധ്യകാല ചോളരുടെ ഉദയത്തിനും ഇടയ്ക്കുള്ള ഇടക്കാലം, വിജയലായന്റെ രാജവംശം, ഒടുവിൽ, കുലോത്തുംഗ ചോളൻ ഒന്നാമൻ മുതൽക്ക്, 11-ആം നൂറ്റാണ്ടിന്റെ മൂന്നാം പാദം വരെ നീണ്ടുനിന്ന പിൽക്കാല ചോള രാജവംശം. സംഘസാഹിത്യത്തിലാണ് വ്യക്തമായ തെളിവുകളുള്ള ചോള

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[7]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[8])

QUESTION    : ചോള സംസ്കാരത്തിൽ പ്രത്യേക സ്ഥാനമുള്ള നദി ഏതായിരുന്നു ?
GOLD ANSWER : ['കാവേരി നദിയ്ക്ക്']
MODEL ANSWER: ്. തോലമൊലി എഴുതിയ സൂല
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] തമിഴ് ഐതിഹ്യങ്ങളനുസരിച്ച്, പുരാതന ചോള രാജ്യത്തിൽ ഉൾപ്പെട്ട പ്രദേശങ്ങൾ ഇന്നത്തെ തമിഴ്നാട്ടിലെ തിരുച്ചിറപ്പള്ളി ജില്ല, തഞ്ചാവൂർ ജില്ല എന്നിവയായിരുന്നു. കാവേരി നദിയും അതിന്റെ ഉപശാഖകളും ഈ പൊതുവെ നിരപ്പാർന്നതും ലഘുവായി സമുദ്രത്തിലേക്ക് ചരിയുന്നതുമായ, വലിയ കുന്നുകളോ താഴ്വാരങ്ങളോ ഇല്ലാത്ത, പ്രദേശത്ത് ഒഴുകി. പൊന്നി’‘ (സ്വർണ്ണ) നദി എന്നും അറിയപ്പെടുന്ന കാവേരി നദിയ്ക്ക് ചോള സംസ്കാരത്തിൽ സവിശേഷ സ്ഥാനമുണ്ടായി

[2] ചോളരുടെ ചരിത്രത്തെ നാലു കാലഘട്ടങ്ങളായി വിഭജിക്കാം: സംഘസാഹിത്യ കാലത്തെ ആദ്യകാല ചോളർ, സംഘം ചോളരുടെ പതനത്തിനും വിജയലായന്റെ കീഴിൽ (ക്രി. വ. 848) മദ്ധ്യകാല ചോളരുടെ ഉദയത്തിനും ഇടയ്ക്കുള്ള ഇടക്കാലം, വിജയലായന്റെ രാജവംശം, ഒടുവിൽ, കുലോത്തുംഗ ചോളൻ ഒന്നാമൻ മുതൽക്ക്, 11-ആം നൂറ്റാണ്ടിന്റെ മൂന്നാം പാദം വരെ നീണ്ടുനിന്ന പിൽക്കാല ചോള രാജവംശം. സംഘസാഹിത്യത്തിലാണ് വ്യക്തമായ തെളിവുകളുള്ള ചോള രാജാക്കന്മാരെക്കുറിച്ചു

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[8]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[9])

QUESTION    : ഹുമയൂൺ തന്റെ മകന് എന്ത് പേരാണ് ആദ്യം പേര് നൽകിയത് ?
GOLD ANSWER : ['ബഹറുദ്ദീൻ (മതപൌർണമി) മുഹമ്മദ് അക്ബർ']
MODEL ANSWER: � ബെംഗലൂരു എന്ന പേരിന
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ഷേർഷായോട് പരാജയപ്പെട്ട് പലായനം ചെയ്ത ഹുമായൂണിന്റെ പ്രവാസകാലത്താണ് അക്ബർ ജനിച്ചത്. അബുൽഫത്ത് ജലാലുദ്ദീൻ മുഹമ്മദ് അക്ബർ എന്നായിരുന്നു പൂർണമായ പേര്. ഹുമയൂൺ തന്റെ പുത്രന് ആദ്യം നല്കിയ പേർ ബഹറുദ്ദീൻ (മതപൌർണമി) മുഹമ്മദ് അക്ബർ എന്നായിരുന്നു. അക്ബർ 'ജൻ' പട്ടണത്തിലെ ഒരു കൊച്ചു വീട്ടിൽ 1543 ജൂലൈ വരെ മാതാവിനോടൊത്തു താമസിച്ചു. കന്ദഹാറിലെത്തിയ ഹുമയൂണിന് അനുജനായ അസ്ക്കാരിയുടെ ശത്രുതമൂലം, അക്ബറെ അവിടെ ഉപേക്ഷിച്ച

[2] അക്കാലത്ത്‌ അത്‌ ഒരു കൃതിയുടെ രൂപം പ്രാപിച്ചോ എന്നത് വ്യക്തമല്ല. ഏറ്റവും കുറഞ്ഞത്‌ ക്രിസ്തുവിനു മുമ്പ്‌ നാനൂറിനും മുന്നൂറിനും ഇടയിലെങ്കിലും മഹാഭാരതം പുസ്തകരൂപം പ്രാപിച്ചു എന്നുമാത്രം മനസ്സിലാക്കാം. വേദകാലത്തെ ഒരു പ്രധാന ഋഷികുടുംബാംഗമായ വസിഷ്ഠമഹർഷിയുടെ പുത്രനായ പരാശരന്‌ ഒരു മുക്കുവസ്ത്രീയിലുണ്ടായ മകനാണ്‌ വ്യാസൻ എന്ന ദ്വൈപായനൻ. അമ്മയെപ്പോലെ തന്നെ കറുത്തനിറമായതിനാൽ കൃഷ്ണദ്വൈപായനൻ 

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[9]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[10])

QUESTION    : 1907 -ൽ, സ്വാതന്ത്ര്യ പ്രസ്ഥാനത്തെ അന്താരാഷ്ട്ര തലത്തിലേക്ക് കൊണ്ടുപോകാൻ ജർമ്മനിയിലെ ഇന്റർനാഷണൽ സോഷ്യലിസ്റ്റ് കോൺഗ്രസിൽ പങ്കെടുക്കാൻ   സവർക്കർ നിയമിച്ചത് ആരെയാണ്?
GOLD ANSWER : ['മാഡം ബിക്കാജി കാമയെ']
MODEL ANSWER: ായ കോളേജിലെ പ്രിൻസിപ
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] അക്കാലത്ത് സ്വാതന്ത്ര്യ സമര സേനാനികളുടെ ഇടയിൽ നിലന്നിന്നിരുന്ന മിതവാദി-തീവ്രവാദി വിഭാഗങ്ങളിൽ രണ്ടാമത്തേതിനൊപ്പമായിരുന്നു സാവർക്കർ. സായുധ സമരത്തിലൂടെ ബ്രിട്ടീഷുകാരെ ഇന്ത്യയിൽനിന്നു പുറത്താക്കണമെന്നാണ് സാവർക്കർ ആഗ്രഹിച്ചത്. 1905ൽ ബംഗാൾ വിഭജനത്തിനെതിരെ നടന്ന ശക്തമായ പ്രക്ഷോഭങ്ങളുടെ ഭാഗമായി വിദേശസാധനങ്ങൾ ബഹിഷ്കരിക്കൽ പ്രക്ഷോഭത്തിൽ സാവർക്കർ ഭാഗഭാക്കായി. അങ്ങനെ സ്വാതന്ത്ര്യ സമരചരിത്രത്തിൽ ആദ്യമായി, പൂനെ

[2] ഇതിൽ സായുധമാർഗ്ഗങ്ങൾ ഉപേക്ഷിച്ച് താൻ ഭരണകൂടത്തിന്റെ ഒപ്പം നിൽക്കാമെന്ന് പറയുന്നുണ്ട്അതേവർഷം തന്നെ അദ്ദേഹം ബ്രിട്ടീഷ് ഭരണകൂടത്തെ അംഗീകരിച്ചുകൊണ്ടും, അക്രമപ്രവർത്തനങ്ങളെ തള്ളിപ്പറഞ്ഞുകൊണ്ടും പ്രസ്താവന നടത്തി. സവർക്കറെ വിട്ടയച്ചാൽ മഹാരാഷ്ട്രയിൽ സ്വാതന്ത്ര്യസമരം വീണ്ടും ആളിപടർന്നേക്കാം എന്

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[10]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[11])

QUESTION    : മൗര്യ കാലത്തു കളക്ടറുടെ ശമ്പളം എത്രയായിരുന്നു?
GOLD ANSWER : ['24,000']
MODEL ANSWER: ��ോഡിക്കുകൾ പുറത്തുകൊ�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] നാട്ടു വാർത്തകൾ ശേഖരിച്ച രാജാവിനടുത്തെത്തിക്കാൻ പ്രത്യേക വിഭാഗം ഉണ്ടായിരുന്നു. രാജ്യ വരുമാനത്തിന്റെ നാലിലൊന്ന് ഉദ്യോഗസ്ഥന്മാർക്ക് ശമ്പളത്തിനും ക്ഷേമപ്രവർത്തനത്തിനും മറ്റുമായി ചെലവാക്കിയിരുന്നു. ഉദ്യോഗത്തിനനുസരിച്ച ശമ്പളം ഏറിയും കുറഞ്ഞുമിരിക്കും. സേനാപതിയ്ക്കും പുരോഹിതനും 48,000 രൂപയായിരുന്നു ശമ്പളം, സമാഹർത്താവിന്റെ വേതനം 24,000 രൂപയായിരുന്നു. പാത നിർമ്മാണം, പൊതു മരാമത്ത് ജലസേചനം തുടങ്ങിയവ ഭരണകൂടം 

[2] വിദേശ വ്യാപാരത്തിന് ചുങ്കം ചുമത്തിയിരുന്നു. ആയുധ നിർമ്മാണം, തോണി-കപ്പൽ നിർമ്മാണം എന്നിവ നികുതിയില്ലാത്തതായിരുന്നു. നൂൽ നൂല്പ്, നെയ്ത്ത്, ഖനനം എന്നിവ ശ്രദ്ധേയമായ വാണിജ്യ മേഖലകൾ ആയിരുന്നു. മിക്കവാറും എല്ലാ ഉത്പന്നങ്ങക്കും നിർമ്മാണ വേളയിലും വില്പന വേളയിലും നികുതി ഒടുക്കേണ്ടീയിരുന്നു. നികുതി വണിക്കുകൾ ഒന്നായി കൊടുക്കേണ്ടതായിരുന്നു. വിലയുടെ അഞ്ചിലൊന്നായിരുന്നു ചുങ്കം. വിദേശിയരുമായുള്ള സമ്പർക്കം നിമിത്ത

[3] അതു വഴി

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[11]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[12])

QUESTION    : മൈസൂർ കടുവ എന്നറിയപ്പെടുന്നത് ആര് ?
GOLD ANSWER : ['ടിപ്പു സുൽത്താന്റെ']
MODEL ANSWER: മൈതാൻ എന്നും ഉത്തര മ�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ഈ കാലഘട്ടത്തിൽ  മഡ്രാസ് പ്രെസിഡൻസിയിൽനിന്ന് തൊഴിലാളികളെ തിരഞ്ഞെടുത്ത് അവരെ സൈനിക മേഖലയിലേക്ക് പുന‍സ്ഥാപിക്കുന്നത് കൂടുതൽ എളുപ്പമാണെന്ന് ബ്രിട്ടീഷുകാർ കണ്ടെത്തി. 1831ൽ മൈസൂർ സാമ്രാജ്യത്തിന്റെ തലസ്ഥാനം മൈസൂരിൽനിന്ന് ബാംഗ്ലൂരിലേക്ക് മാറ്റപ്പെട്ടു. ഈ കാലത്തുണ്ടായ രണ്ട് പുരോഗതികൾ നഗരത്തിന്റെ വേഗത്തിലുള്ള വളർച്ചക്ക് കാരണമായി. ടെലിഗ്രാഫിന്റെ ഉപയോഗവും 1864ൽ മദിരാശിയുമായി ബന്ധിപ്പിച്ച റെയിൽ മാർഗവുമായിരുന്ന

[2] തെക്കേ ഇന്ത്യൻ  സംസ്ഥാനമായ കർണാടകയുടെ തെക്ക് കിഴക്ക് ഭാഗത്താണ്‌ ബാംഗ്ലൂർ. മൈസൂർ പീഠഭൂമിയുടെ ഹൃദയഭാഗത്തു സ്ഥിതി ചെയ്യുന്ന ഈ നഗരം സമുദ്രനിരപ്പിൽ നിന്നും 920 മീറ്റർഉയരത്തിലാണു  (3,018  അടി)സ്ഥിതി ചെയ്യുന്നത് . 12.97°N 77.56°E﻿ / 12.97; 77.56 എന്ന സ്ഥാനത്ത് സ്ഥിതി ചെയ്യുന്ന ഈ നഗരത്തിന്റെ വിസ്തീർണ്ണം 741 കിലോമീറ്റർ² (286 മൈൽ²). . നഗരത്തിന്റെ ഭൂരിഭാഗവും ബാംഗ്ലൂർ അർബൻ ജില്ലയിലാണ്‌ . പരിസര പ്രദേശങ്ങൾ ബാംഗ

[3] പ്രാദേശ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[12]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[13])

QUESTION    : ഗൂർഖയിലെ പൃഥ്വി നാരായൺ ഷാ നേപ്പാളിനെ ഏകീകരിച്ചത് ഏത് വർഷം?
GOLD ANSWER : ['1765ൽ']
MODEL ANSWER: �ാമ്രാജ്യത്തിന്റെ പ്രഭ�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] കാഠ്മണ്ഡു താഴ്വരയിൽ നിന്നു ലഭിച്ച നവീന ശിലായുഗ ആയുധങ്ങൾ ഇവിടെ ഒമ്പതിനായിരം വർഷം മുൻപ് ഇവിടെ ജനവാസമുണ്ടായിരുന്നു എന്ന് സൂചിപ്പിക്കുന്നു. രണ്ടായിരത്തിയഞ്ഞൂറ് വർഷം മുൻപ് ഇവിടെ ടിബറ്റോ-ബർമൻ വംശജർ ഇവിടെ താമസിച്ചിരുന്നതായി കണക്കാക്കുന്നു. ‍ഇന്തോ-ആര്യൻ ഗോത്രക്കാർ ഇവിടെ ഏകദേശം ൧൫൦൦(1500)ബി. സി. യിൽ എത്തിയതായി കണക്കാക്കുന്നു.  ൧൦൦൦(1000)ബി. സി. യോടടുത്ത് ചെറിയ നാട്ടുരാജ്യങ്ങൾ രൂപപ്പെട്ടു തുടങ്ങി.  ൨൫൦(250)

[2] ശേഷം ഗ്യാനേന്ദ്ര മഹാരാജാവ് സ്ഥാനമേറ്റു. ൨൦൦൨(2002)ൽ ‍രാജാവ് പാർലമൻറ് പിരിച്ചുവിട്ടു. 2005ൽ അദ്ദേഹം അടിയന്തരാവസ്ഥ പ്രഖ്യാപിക്കുകയും ഭരണം പൂർണമായി സ്വയം ഏറ്റെടുക്കുകയും ചെയ്തു. 2006ലെ ജനാധിപത്യ നീക്കത്തെ തുടർന്നു ഏപ്രിൽ 24,2006 ന് പരമാധികാരം ജനങ്ങൾക്കു തിരിച്ചു നൽകുകയും പാർലമെൻറ് വീണ്ടും വിളിച്ചുകൂട്ടുകയും ചെയ്തു. മെയ് 18, 2006ന് പാർലമെൻറ് നേപാളിനെ മതേതര രാഷ്ട്രമായി പ്രഖ്യാപിച്ചു. 1996 മുതൽ 2006 വരെ


In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[13]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[14])

QUESTION    : ആന്ധ്ര സംസ്ഥാനത്തിന്റെ തലസ്ഥാനം ഏത് ?
GOLD ANSWER : ['ഹൈദരാബാദിനു']
MODEL ANSWER: �ിഡിയുടെ സാമ്രാജ്യത്ത
Gold in retrieved passages: False

RETRIEVED PASSAGES:
[1] ഈ കാലഘട്ടത്തിൽ  മഡ്രാസ് പ്രെസിഡൻസിയിൽനിന്ന് തൊഴിലാളികളെ തിരഞ്ഞെടുത്ത് അവരെ സൈനിക മേഖലയിലേക്ക് പുന‍സ്ഥാപിക്കുന്നത് കൂടുതൽ എളുപ്പമാണെന്ന് ബ്രിട്ടീഷുകാർ കണ്ടെത്തി. 1831ൽ മൈസൂർ സാമ്രാജ്യത്തിന്റെ തലസ്ഥാനം മൈസൂരിൽനിന്ന് ബാംഗ്ലൂരിലേക്ക് മാറ്റപ്പെട്ടു. ഈ കാലത്തുണ്ടായ രണ്ട് പുരോഗതികൾ നഗരത്തിന്റെ വേഗത്തിലുള്ള വളർച്ചക്ക് കാരണമായി. ടെലിഗ്രാഫിന്റെ ഉപയോഗവും 1864ൽ മദിരാശിയുമായി ബന്ധിപ്പിച്ച റെയിൽ മാർഗവുമായിരുന്ന

[2] മഹാഭാരതത്തിലും ഐതെരീയ ബ്രാഹ്മണ ഇതിഹാസത്തിലും ആന്ധ്രാ രാജ്യത്തെ പരാമർശിച്ചിട്ടുണ്ട്. മൌര്യരാജാക്കൻമാരുടെ കാലത്തും ആന്ധ്ര എന്ന രാജ്യം ഉണ്ടായിരുന്നതായി ചരിത്രകാരൻമാർ രേഖപ്പെടുത്തിയിട്ടുണ്ട്. ബി. സി ഒന്നാം നൂറ്റാണ്ടിൽ എഴുതപ്പെട്ട ഭരതന്റെ നാട്യശാസ്ത്രത്തിലും "ആന്ധ്രാ വംശത്തെ" കുറിച്ച് പരാമർശം ഉണ്ട്. ഗുണ്ടൂർ ജില്ലയിലെ ഭട്ടിപ്റോലു ഗ്രാമത്തിൽ കാണുന്ന ലിഖിതങ്ങൾ തെലുങ്ക് ഭാഷയുടെ വേരുകളിലേക്കു വഴിതെളിക്കുന്ന

[3] ബ്രിട്ടീഷ് 

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[14]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[15])

QUESTION    : ഗ്വാളിയോർ യുദ്ധത്തിനു ശേഷം ടാൻടിയോ തോപ്പി തോപീ എവടക്കാന് പലായനം ചെയ്തത് ?
GOLD ANSWER : ['രാജ്പുതാനയിലേക്ക്']
MODEL ANSWER: ��ാഠ്മണ്ഡു താഴ്വരയിൽ ന
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ഗ്വാളിയോർ യുദ്ധത്തിനു ശേഷം താന്തിയോ തോപ്പെ, നാനാ സാഹിബിന്റെ അനന്തരവനായ റാവു സാഹിബിനുമൊപ്പം രാജ്പുതാനയിലേക്ക് പലായനം ചെയ്തു. സൈന്യത്തെ സംഘടിപ്പിച്ച് ബ്രിട്ടീഷുകാർക്കെതിരെ യുദ്ധം ചെയ്യുക എന്നതായിരുന്നു ഇരുവരുടേയും പദ്ധതി. എന്നാൽ ബ്രിട്ടീഷ് പട്ടാളം ഇവരെ വളയുകയും പിടികൂടാൻ ശ്രമിക്കുകയും ചെയ്തെങ്കിലും താന്തിയോ തോപ്പെ യുദ്ധമുഖത്തു നിന്നും ഉദയ്പൂർ ലക്ഷ്യമാക്കി കടന്നു കളഞ്ഞു. റാവു സാഹീബ് തോപ്പെയേ അനുഗമിക്

[2] എന്നിരിക്കിലും സുരക്ഷിതമായ ഒരു താവളത്തിലെത്താതെ റാണി വിശ്രമിക്കാൻ തയ്യാറായില്ല. 24 മണിക്കൂറുകൊണ്ട് റാണിയും സംഘവും ഏതാണ്ട് 86 മൈലുകൾ സഞ്ചരിക്കുകയുണ്ടായി. അർദ്ധരാത്രിയോടെ കല്പിയിൽ സുരക്ഷിതമായ ഒരു താവളത്തിലെത്തുകയും ചെയ്തു. റാണിയുടെ പെട്ടെന്നുണ്ടായ വരവിൽ ആശ്ചര്യം പൂണ്ടെങ്കിലും റാവോ സാഹേബ് അവരെ സന്തോഷപൂർവ്വം സ്വീകരിച്ചു. പിന്നീട് താന്തിയോ തോപ്പേയും ഇവരോടൊപ്പം ചേരുകയുണ്ടായി. റാ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[15]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[16])

QUESTION    : മുഗൾ കൊട്ടാരങ്ങളിൽ ചക്രവർത്തി സാധാരണക്കാരുമായി ഇടപഴകുന്നത്  ഏത് ഹാളിൽ വച്ചാണ് ?
GOLD ANSWER : ['ദിവാൻ ഇ-ആം']
MODEL ANSWER: �ുകൊണ്ടുള്ള പ്രഭുക്കന്മ
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] പുരാനി ദില്ലി അഥവാ ഷാജഹാനാബാദ് നഗരത്തിന്റെ കേന്ദ്രഭാഗമാണ് ചെങ്കോട്ട. കോട്ടയുടെയും നഗരത്തിന്റെയും കിഴക്കുവശം യമുനാനദിയാണ്. പടിഞ്ഞാറുവശത്തുള്ള ലാഹോറിഗേറ്റ്, തെക്കുവശത്തുള്ള ഡെൽഹി ഗേറ്റ് എന്നീ രണ്ട് പ്രധാനപ്രവേശനകവാടങ്ങൾ കോട്ടക്കുണ്ട്. ഈ കവാടങ്ങളിൽ നിന്നുള്ള വഴികൾ ചെന്നെത്തുന്ന നഗരമതിലിലെ കവാടങ്ങൾക്കും ഇതേ പേരുകൾ തന്നെയാണ്. യമുനയിലേക്കിറങ്ങുന്ന രാജ്ഘാട്ട് ഗേറ്റ് എന്ന കവാടവും കോട്ടക്കുണ്ട്. പടിഞ്ഞാറുവ

[2] 1803-ലെ ദില്ലി യുദ്ധത്തിൽ മറാഠരെ പരാജയപ്പെടുത്തി ബ്രിട്ടീഷുകാർ ഉത്തരേന്ത്യയുടെ നിയന്ത്രണം കൈയടക്കി. മുഗളരുടെ സംരക്ഷകരായി ദില്ലിയിലെത്തിയ ബ്രിട്ടീഷുകാർ തുടക്കത്തിൽ ചക്രവർത്തിയോട് ബഹുമാനപൂർവ്വമായിരുന്നു പെരുമാറിയിരുന്നത്. അവർ നാണയങ്ങൾ ചക്രവർത്തിയുടെ പേരിലായിരുന്നു അടിച്ചിറക്കിയിരുന്നത്. കമ്പനിയുടെ സീലിൽപ്പോലും മുഗൾ ചക്രവർത്തി ഷാ ആലത്തെ അംഗീകരിച്ചുകൊണ്ട് ഫിദ്വി ഷാ ആലം (ഷാ ആലത

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[16]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[17])

QUESTION    : മന്ത്രിസഭ സമുദ്രഗുപ്തനു നൽകിയ  പേരെന്ത് ?
GOLD ANSWER : ['കവിരാജൻ']
MODEL ANSWER: ന്റെ പ്രധാന സംഘടനകളും �
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] എന്നാൽ അപ്രതീക്ഷിതമായി ജയലളിതയുടെ നേതൃത്വത്തിലുള്ള ഓൾ ഇന്ത്യാ അണ്ണാ ദ്രാവിഡ മുന്നേറ്റ കഴകം വാജ്പേയി സർക്കാരിനുള്ള പിന്തുണ പിൻവലിക്കുകയാണെന്നു കാണിച്ച് രാഷ്ട്രപതിക്കു കത്തു നൽകി. വാജ്പേയിയോട് സഭയിൽ വിശ്വാസവോട്ടു തേടാൻ രാഷ്ട്രപതി ആവശ്യപ്പെട്ടു. ഇതനുസരിച്ച് പ്രവർത്തിച്ച വാജ്പേയിയുടെ നേതൃത്വത്തിലുള്ള എൻ. ഡി. എ. സർക്കാർ സഭയിൽ പരാജയപ്പെട്ടു. തുടർന്ന കോൺഗ്രസ്സും, ബി. ജെ. പിയും മന്ത്രിസഭാ രൂപീകരണത്തിനു ശ്

[2] മാത്രവുമല്ല, ദക്ഷിണോത്തര ഭാഗങ്ങൾ തമ്മിലുള്ള ഗതാഗതം സുഗമമാക്കുവാനായി സാമ്രാജ്യത്തിലെ ദക്ഷിണ ഭാഗങ്ങളിലെ കാട്ടു പ്രദേശങ്ങൾ ആക്രമിച്ച് കീഴടക്കുകയൂം ചെയ്തു. കിഴക്കൻ അതിർത്തിയിലെ നേപ്പാളം, സമതടം, കാർത്രീപുത്രം, കാമരൂപം എന്നീ രാജ്യങ്ങളും ഗിരിവർഗ്ഗക്കാരായ മാളവർ, യൌധേയർ, മാദ്രകർ, ആഭീരന്മാർ എന്നിവരും സമുദ്രഗുപ്തന്റെ മേൽക്കോയ്മ അംഗീകരിച്ചിരുന്നു. ശ്രീലങ്ക യിൽ നിന്നും അവിടത്തെ രാജാക്കന്മർ അദ്ദേഹത്തിന് കപ്പം ന

[3] 1947 ആഗസ്റ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[17]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[18])

QUESTION    : ജവഹർ ലാൽ നെഹ്‌റു ,സമ്പൂർണ്ണ സ്വാതന്ത്ര്യം എന്ന ആശയം മുന്നോട്ടുവച്ചത് ഏത് വര്ഷം ?
GOLD ANSWER : ['1927 ൽ']
MODEL ANSWER: ൾ അദ്ദേഹം അദ്ദേഹം സ്വാ
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] പതിമൂന്നാം വയസ്സിൽ കുടുംബസുഹൃത്തായിരുന്ന  ആനീബസന്റിന്റെ കൂടെ തിയോസഫിക്കൽ സൊസൈറ്റിയിൽ നെഹ്രു അംഗമായി. തന്നെ ഏറെ സ്വാധീനിച്ച ബ്രൂക്ക്സുമായി വേർപിരിഞ്ഞതോടെ നെഹ്രു തിയോസഫിക്കൽ സൊസൈറ്റിയിൽ നിന്നും വിടുതൽ നേടി. ബ്രിട്ടീഷ് ഇന്ത്യയിലെ ഏറ്റവും മികച്ച വിദ്യാലയങ്ങളിൽ പ്രാഥമിക വിദ്യാഭ്യാസം നേടിയ ജവഹർലാൽ, ഉപരിപഠനത്തിനായി ഇംഗ്ലണ്ടിലേക്ക്‌ പോയി . ഇംഗ്ലണ്ടിലെ ഹാരോസ്കൂൾ, കേംബ്രിഡ്ജ്‌ -ട്രിനിറ്റി കോളജ് എന്നിവിടങ്ങള

[2] സ്വാതന്ത്ര്യം ലഭിക്കുന്നതിനുമുൻപ് അദ്ദേഹം ബോംബെ യുടെ ആഭ്യന്തരമന്ത്രിയായിരുന്നു. സ്വാതന്ത്ര്യലബ്ധിക്കുശേഷം അദ്ദേഹം 1952-ൽ ബോംബെ സംസ്ഥാനത്തിന്റെ മുഖ്യമന്ത്രിയായി തിരഞ്ഞെടുക്കപ്പെട്ടു. മറാഠി ഭാഷയും, ഗുജറാത്തി ഭാഷയും സംസാരിക്കുന്ന ജനങ്ങൾ തിങ്ങിപ്പാർത്ത ബോംബെ സംസ്ഥാനത്ത് ഇരു വിഭാഗങ്ങളും ഭാഷാടിസ്ഥാനത്തിൽ പ്രത്യേക സംസ്ഥാനം ആവശ്യപ്പെട്ടു. ഒരു ശക്തനായ നേതാവായി അറിയപ്പെട്ട മൊറാർജി ചിലപ്

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[18]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[19])

QUESTION    : ഹനുമാൻ തന്റെ ദൂതനാണെന്നതിന് സീതക്ക് തെളിവ് കാണിക്കാൻ രാമൻ എന്താണ്  ഹനുമാനു കൈമാറിയത് ?
GOLD ANSWER : ['മുദ്രമോതിരം']
MODEL ANSWER: ം സീതയെ അന്വേഷിക്കുന്ന
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ശക്തിയുണ്ടെങ്കിൽ അഗ്നിദേവൻ ഹനുമാന് കുളിർമ്മയേകട്ടെ എന്നു  പ്രാർത്ഥിച്ചു. ലങ്ക ആകമാനം വീക്ഷിച്ച ഹനുമാൻ ഒരാട്ടഹാസത്തോടെ തന്നെ ബന്ധിച്ച കയർ പൊട്ടിച്ചു , മാളികകളിൽ  നിന്നും മാളികകളിലേക്കു ചാടി ലങ്കാനഗരത്തിന് തീവെച്ചു. ലങ്കാനഗരത്തിൽ മുഴുവൻ ഹനുമാൻ തീ പടർത്തിയെങ്കിലും  വീഭിഷണന്റെ ഗൃഹത്തിന് ഹനുമാൻ തീ വെച്ചില്ല. ലങ്കയിൽ ആളിപടർന്ന തീ , സീത വസിക്കുന്ന അശോകവനത്തിലും എത്തിയില്ല. സമുദ്രത്തിൽ വാൽ മുക്കി തീ അണച്ച

[2] സുഗ്രീവനെ നിഷ്കരുണം രാജ്യത്തിൽ നിന്നും നിഷ്കാസനം ചെയ്തു അദ്ദേഹത്തിന്റെ പത്നിയെ തട്ടിയെടുത്ത ക്രൂരനായ ജ്യേഷ്ഠനും കിഷ്കിന്ധയുടെ രാജാവുമായ ബാലിയെ വധിച്ചു സുഗ്രീവനെ രാജാവായി വാഴിക്കാമെന്നു രാമനും പകരം തന്റെ സേനയെ ഉപയോഗിച്ചു സീതയെ കണ്ടെത്തി വീണ്ടെടുക്കാൻ സഹായിക്കാമെന്ന് സുഗ്രീവനും പരസ്പരം വാക്കു കൊടുത്തു. ആ അവസരത്തിൽ രാവണന്റെയും ബാലിയുടെയും സീതയുടെയും ഇടത്തെ കണ്ണു അടിക്ക

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[19]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[20])

QUESTION    : മാതൃരാജ്യമായ അൽബേനിയൻ  പ്രവേശിക്കാൻ   മദർ  തെരേസയ്‌ക്കു  അനുമതി നിഷേധിച്ചപ്പോൾ അവിടത്തെ സർക്കാരിൽ  സമ്മർദ്ദം ചെലുത്തി ഇന്ത്യൻ പ്രധാനമന്ത്രി ആരായിരുന്ന് ?
GOLD ANSWER : ['ഇന്ദിരാ ഗാന്ധി']
MODEL ANSWER: ന്നതിനുള്ള പ്രതിഫലം മദർ �
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] 1971 ലെ ബംഗ്ലാദേശ് വിമോചനയുദ്ധത്തിൽ ഏതാണ്ട് മുപ്പതു ലക്ഷത്തോളം  ജനങ്ങൾ മരിക്കുകയുണ്ടായി. അതിലേറെപ്പേർക്ക് സ്വഭവനങ്ങളടക്കം തങ്ങൾക്കുണ്ടായിരുന്നതെല്ലാം നഷ്ടപ്പെട്ടു. കൂടാതെ പാകിസ്താൻ പട്ടാളക്കാർ ഏതാണ്ട് 4,000 ഓളം സ്ത്രീകളെ ബലാത്സംഗത്തിനിരയാക്കിയിരുന്നു എന്നും രേഖകൾ പറയുന്നു. അങ്ങനെ ബംഗ്ലാദേശ് യുദ്ധക്കെടുതിയിൽ നരകിക്കുകയായിരുന്നു. ഈ സമയത്ത് മദർ തെരേസ ബംഗ്ലാദേശിലെ പീഡനത്തിനിരയായ സ്ത്രീകളെ സഹായിക്കാനായ

[2] 1946 സെപ്റ്റംബർ 10-നു വാർഷികധ്യാനത്തിനായി ഡാർജിലിങ്ങിലെ ലൊറേറ്റോ കോൺ‌വെന്റിലേക്കുള്ള യാത്രാമധ്യേ ആണ് തെരേസ തന്റെ സന്യാസജീവിതത്തിന്റെ ദിശമാറ്റിവിടാനുള്ള ഉറച്ച തീരുമാനത്തിലെത്തുന്നത്. ലൊറെറ്റോ സഭ വിട്ടിറങ്ങി പാവങ്ങൾക്കൊപ്പം ജീവിച്ച് അവരെ സേവിക്കുക എന്നതായിരുന്നു തെരേസ എടുത്ത തീരുമാനം. ദൈവവിളിക്കു

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[20]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[21])

QUESTION    : ടാഗോറിന്റെ ആത്മകഥ മലയാളത്തിലേക്ക് വിവർത്തനം ചെയ്തത് ആര് ?
GOLD ANSWER : ['കൃഷ്ണ കൃപാലാനി']
MODEL ANSWER: �. ഇതിനെ പേരിലേക്ക് പ�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ഒട്ടു മിക്ക മുഖ്യകൃതികളും വിവർത്തനം ചെയ്തുകൊണ്ട് മലയാളസാഹിത്യം ടാഗൂറിനെ ആദരിച്ചിട്ടുണ്ട്. വിവർത്തന  സംരംഭങ്ങൾക്കു തുടക്കം കുറിച്ചത് സഞ്ജയൻ ആയിരുന്നു. ശങ്കരക്കുറുപ്പ്, പുത്തേഴത്തു രാമൻ മേനോൻ, വി. ഉണ്ണിക്കൃഷ്ണൻ നായർ എന്നിവരിലൂടെയാണ് വിവർത്തനങ്ങൾ ഏറെയും പുറത്തിറങ്ങിയത്. ഗീതാഞ്ജലിക്കു പദ്യത്തിലും ഗദ്യത്തിലുമായി ഒട്ടനവധി പരിഭാഷകൾ ഉണ്ടായിട്ടുണ്ടെങ്കിലും ജി. യുടെ കൃതിക്കാണ് ഏറെ സ്വീകാര്യത ഉണ്ടായത്. ഗീതാഞ

[2] 1901ൽ ടാഗോർ ഷിലൈധ വിട്ട്‌ പശ്ചിമ ബംഗാളിലെ ശാന്തിനികേതനത്തിൽ താമസമാരംഭിച്ചു. അവിടെ തറയിൽ വെണ്ണക്കല്ല് പതിച്ച പ്രാർഥന മുറിയുള്ള ഒരു ആശ്രമവും പരീക്ഷണാടിസ്ഥാനത്തിൽ ഒരു വിദ്യാലയവും, പുഷ്പ-വൃക്ഷ തോട്ടങ്ങളും, ഒരു വായനശാലയും അദ്ദേഹം സ്ഥാപിച്ചു. ഇവിടെ വച്ച്‌ ടാഗോറിന്റെ ഭാര്യയും(1902-ൽ) രണ്ട്‌ കുട്ടികളും മരണമടഞ്ഞു. അദ്ദേഹത്തിന്റെ പിതാവ്‌ 19 ജനുവരി 1905ൽ മരണമടഞ്ഞു. ടാഗൂറിനു വ്യക്തിപരമായി അളവറ്റ ദുഃഖം ഉ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[21]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[22])

QUESTION    : 1906 ശരത്കാലത്ത് ന്യൂയോർക്ക് സിറ്റിയിലെ 1 വെസ്റ്റ് 34 സ്ട്രീറ്റിൽ, ബറകത്തുള്ളയും സാമുവൽ ലൂക്കോസ് ജോഷിയും ചേർന്ന് രൂപീകരിച്ച സംഘടനഏത് ?
GOLD ANSWER : ['പാൻ-ആര്യൻ അസോസിയേഷൻ']
MODEL ANSWER: ന്റെ പ്രകാരം സാവർക്കർ സ
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] അവിടെ ടോക്കിയോ സർവകലാശാലയിൽ ഹിന്ദുസ്ഥാനി പ്രൊഫസറായി നിയമിതനായി. 1906 ലെ ശരത്കാലത്തിലാണ്, ന്യൂയോർക്ക് നഗരത്തിലെ 1 വെസ്റ്റ് 34 സ്ട്രീറ്റിൽ, ബരകത്തുല്ലയും പരേതനായ റെവറന്റ് ലൂക്കാസ് മലോബ ജോഷിയുടെ മകനായ മറാത്ത ക്രിസ്ത്യാനിയായ സാമുവൽ ലൂക്കാസ് ജോഷിയും ചേർന്ന് പാൻ-ആര്യൻ അസോസിയേഷൻ രൂപീകരിച്ചു പ്രവർത്തിച്ചു. ക്ലാൻ-നാ-ഗെയ്‌ലിന്റെ ഐറിഷ് വിപ്ലവകാരികൾ, ബ്രിട്ടീഷ് വിരുദ്ധ അഭിഭാഷകൻ മൈറോൺ എച്ച്. ഫെൽപ്‌സ്, സ്വാമി വ

[2] ലാൽ‌ബാഗ്  -ബാംഗ്ലൂരിലെ പ്രധാനപ്പെട്ട ഒരു ഉദ്യാനമാണ്‌ ലാൽബാഗ്. മൈസൂർ ഹൈദർ അലി എന്ന അധികാരി ആയിരുന്നു ഈ ഉദ്യാനം കമ്മീഷൻ ചെയ്തത്, എങ്കിലും ഇത് പൂർത്തിയാക്കിയത് അദ്ദേഹത്തിനെ മകൻ ടിപ്പു സുൽത്താൻ ആണ്‌. 240 ഏക്കറിലധികം പരന്നുകിടക്കുന്ന ഈ ഉദ്യാനം 1000 ലധികം സസ്യവർഗ്ഗങ്ങളുടെ ഒരു സംഭരണസ്ഥലമാണ്‌. ഈ ഉദ്യാനത്തിൽ എല്ലാ വർഷവ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[22]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[23])

QUESTION    : രാജ്യത്തെ എയർപോർട്ടുകൾ നിയന്ത്രിക്കുന്നത് ആരാണ്?
GOLD ANSWER : ['ഇന്ത്യൻ വിമാനത്താവള അതോറിറ്റി(']
MODEL ANSWER: ഫീക്കുകൾ പുറത്തുള്ള സ്റ്�
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ദക്ഷിണേന്ത്യൻ ദേശീയപാതകളുടെ മധ്യസ്ഥാനത്തായി വരാണസി-കന്യാകുമാരി ദേശീയപാതയിൽ സ്ഥിതിചെയ്യുന്നു. കേരളത്തിൽ നിന്നും കാസറഗോഡ് - പുത്തൂർ - ഹാസൻ വഴിയോ കോഴിക്കോട്,സുൽത്താൻ ബത്തേരി - മൈസൂർ ദേശീയപാത 212 വഴിയോ   സേലം വഴിയോ (ദേശീയപാത 544)കാസർഗോഡ് - മൈസൂർ വഴിയോ  ഇരിട്ടി, കൂട്ടുപുഴ, വീരാജ് പേട്ട, മൈസൂർ (തലശ്ശേരി - കുടഗ് സംസ്ഥാനപാത) വഴിയോ   പെരിന്തൽമണ്ണ-നിലമ്പൂർ- ഗൂഡല്ലൂർ-മൈസൂർ വഴിയോ എത്തിച്ചേരാവുന്നതാണ്‌. ബാഗ്ലൂ

[2] ചോളരുടെ ചരിത്രത്തെ നാലു കാലഘട്ടങ്ങളായി വിഭജിക്കാം: സംഘസാഹിത്യ കാലത്തെ ആദ്യകാല ചോളർ, സംഘം ചോളരുടെ പതനത്തിനും വിജയലായന്റെ കീഴിൽ (ക്രി. വ. 848) മദ്ധ്യകാല ചോളരുടെ ഉദയത്തിനും ഇടയ്ക്കുള്ള ഇടക്കാലം, വിജയലായന്റെ രാജവംശം, ഒടുവിൽ, കുലോത്തുംഗ ചോളൻ ഒന്നാമൻ മുതൽക്ക്, 11-ആം നൂറ്റാണ്ടിന്റെ മൂന്നാം പാദം വരെ നീണ്ടുനിന്ന പിൽക്കാല ചോള രാജവംശം. സംഘസാഹിത്യത്തിലാണ് വ്യക്തമായ തെളിവുകളുള്ള ചോള രാജാക്കന്മാ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[23]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[24])

QUESTION    : 1857 ലെ ഡൽഹി കലാപം കാൺപൂരിലേക്ക് വ്യാപിക്കുകയാണെങ്കിൽ, ബ്രിട്ടീഷ് സൈന്യത്തെ സഹായിക്കാൻ തന്റെ നേതൃത്വത്തിൽ 15,000 സൈനികരെ എടുക്കുമെന്ന് നാനാ സാഹിബ് ആർക്കാണ് ഉറപ്പ് നൽകിയത് ?
GOLD ANSWER : ['ചാൾസ് ഹില്ലേഴ്സന്']
MODEL ANSWER: �ുതൽ 1860 വരെ ജീവിച്ചു.
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] 1857 ൽ ഡെൽഹിയിൽ പൊട്ടിപ്പുറപ്പെട്ട കലാപം കാൺപൂരിലേക്കും വ്യാപിക്കുകയാണെങ്കിൽ തന്റെ അധീനതയിലുള്ള 15000 പടയാളികളെ ബ്രിട്ടീഷ് സൈന്യത്തിന്റെ സഹായത്തിനായി എത്തിച്ചുകൊള്ളാം എന്ന് നാനാ സാഹിബ് കാൺപൂർ കലക്ടറായിരുന്ന ചാൾസ് ഹില്ലേഴ്സന് ഉറപ്പു നൽകിയിരുന്നു. കലാപകാരികൾ കാൺപൂരിൽ എത്തിയപ്പോൾ നേരത്തേ വാഗ്ദാനം ചെയ്തിരുന്നപോലെ നാനാ സാഹിബിന്റെ പടയാളികൾ ബ്രിട്ടീഷുകാർക്ക് സഹായവുമായി എത്തിച്ചേർന്നു. അപ്രതീക്ഷിതമായി തന്

[2] കുട്ടികളെ കൂടാതെ ഈ താവളത്തിലുണ്ടായിരുന്ന ആകെ സ്ത്രീകളുടെ എണ്ണം 200 ഓളം വരുമായിരുന്നു. സ്ത്രീകളും കുട്ടികളും ഉൾപ്പെടുന്ന തടവുകാരെ ഒരു മനുഷ്യകവചമാക്കി നിർത്തി ഈസ്റ്റ് ഇന്ത്യാ കമ്പനിയോട് വിലപേശാം എന്നതായിരുന്നു നാനാ സാഹിബിന്റെ പദ്ധതി. കാൺപൂർ പിടിച്ചെടുക്കാൻ ജനറൽ ഹെൻട്രി ഹാവെലോക

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[24]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[25])

QUESTION    : പതിനേഴാം നൂറ്റാണ്ടിൽ മുഗൾ സാമ്രാജ്യത്തിന്റെ നിലനിൽപ്പിനെ ചോദ്യം ചെയ്തത്  എന്ത് ?
GOLD ANSWER : ['കർഷകരുടെ ഇത്തരം എതിർപ്പുകൾ']
MODEL ANSWER: �ബർ സാമ്രാജ്യം സ്ഥാപി
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] പതിനേഴാം നൂറ്റാണ്ടിന്റെ അവസാനമായപ്പോഴേക്കും ധാരാളം ധനവും വിഭവങ്ങളും കൈകാര്യം ചെയ്തിരുന്ന പ്രഭുക്കന്മാർ വളരെ ശക്തിശാലികളായി. കാലക്രമേണ മുഗൾ ചക്രവർത്തിക്ക് സ്വാധീനം കുറഞ്ഞതോടെ ചക്രവർത്തിയുടെ സേവകർ അവരവരുടെ മേഖലകളിൽ ശക്തമായ അധികാരകേന്ദ്രങ്ങളായി പരിണമിച്ചു. ഹൈദരബാദ്, അവാധ്, ബംഗാൾ തുടങ്ങിയ ഇടങ്ങളിലെ അധികാരം അവർ സ്വതന്ത്രമായി പിടീച്ചെടുത്ത് സാമ്രാജ്യങ്ങൾ സ്ഥാപിച്ചു. എന്നിരുന്നാലും ദില്ലിയിലെ മുഗൾ ചക്രവർ

[2] അക്ബറുടെ കാലത്ത് ജഗീറുകളിൽ നിന്നു പിരിക്കുന്ന നികുതിയും മാൻസബ്ദാറുകൾക്കുള്ള വേതനവും ഒരു തുലനാവസ്ഥയിലായിരുന്നു. എന്നാൽ ഔറംഗസേബിന്റെ കാലമായപ്പോഴേക്കും മാൻസബ്ദാറുകളുടെ എണ്ണം ഗണ്യമായി വർദ്ധിച്ചു. 5000-ത്തിലധികം സാറ്റ് സംഖ്യയുള്ള 29 മാൻസബ്ദാറുകൾ മാത്രമേ അക്ബർക്കു കീഴിലുണ്ടായിരുന്നെങ്കിൽ ഔറംഗസേബിന്റെ കാലമായപ്പോഴേക്കും ഇത് 29-ൽ നിന്ന് 79 ആയി വർദ്ധിച്ചു. ഇവർക്കനുവദി

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[25]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[26])

QUESTION    : കൊൽക്കത്തയിൽ ഹിന്ദു-മുസ്ലീം കലാപം ഉണ്ടായ വര്ഷം ഏത് ?
GOLD ANSWER : ['1946 ഓഗസ്റ്റ് 16']
MODEL ANSWER: ദ്യം ഇന്ത്യയിലെ പാവപ്പെ
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] 1857-ൽ ഇംഗ്ലീഷ്‌ ഈസ്റ്റിന്ത്യാ കമ്പനിക്കു നേരെയുണ്ടായ കലാപമാണ്‌ യൂറോപ്യൻ അധിനിവേശത്തിനു നേരെ ഇന്ത്യക്കാർ നടത്തിയ പ്രധാന ചെറുത്തുനിൽപ്പ്‌ ശ്രമം. ഒന്നാം ഇന്ത്യൻ സ്വാതന്ത്ര്യ സമരം എന്നറിയപ്പെടുന്ന ഈ കലാപം പക്ഷേ ബ്രിട്ടീഷ്‌ സൈന്യം അടിച്ചൊതുക്കി. ഇന്ത്യയുടെ ഭൂരിഭാഗം പ്രദേശങ്ങളും ബ്രിട്ടീഷ്‌ സാമ്രാജ്യത്തിനു കീഴിലുമായി. ഇരുപതാം നൂറ്റാണ്ടിന്റെ തുടക്കത്തിൽ മഹാത്മാ ഗാന്ധിയുടെ നേതൃത്വത്തിൽ അഹിംസയിൽ അധിഷ്ഠിതമ

[2] ഇന്ത്യയിലെ പശ്ചിമബംഗാളിന്റെ ചരിത്രമാണ് ബംഗ്ലാദേശിന്റെ ആദ്യകാലചരിത്രം. "ഭിന്നിപ്പിച്ചു ഭരിക്കുക" എന്ന ബ്രിട്ടീഷുകാരുടെ തന്ത്രത്തിന്റെ ഫലമായി 1905-ൽ കഴ്സൺ പ്രഭു ബംഗാളിനെ കിഴക്കൻ ബംഗാളെന്നും പടിഞ്ഞാറൻ ബംഗാളെന്നും വിഭജിച്ചുവെങ്കിലും ജനരോഷത്തെത്തുടർന്ന് വിഭജനം റദ്ദാക്കാൻ ബ്രിട്ടൺ നിർബന്ധിതമായി. ധാക്ക കേന്ദ്രമാക്കി കിഴക്കൻ ബംഗാളിനെ സൃഷ്ടിക്കുകയെന്നായിരുന്നു ബ്രിട്ടീഷുകാരുടെ ലക്ഷ്യം. മുസ്ലീങ്ങൾക്ക് ഭ

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[26]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "retrieval_missed"
})

In [ ]:
show_case_ml(sample_questions_ml[27])

QUESTION    : കൃത്യസമയത്തു പണി കഴിയില്ല എന്ന് സാമന്തരയർ അറിയാച്ചിതിനാൽ ക്ഷേത്രനിർമ്മാണജോലി രാജാവ് ഏൽപ്പിച്ചത് ആരെയായിരുന്നു ?
GOLD ANSWER : ['ബിസു മഹാറാണയെ']
MODEL ANSWER: �ശിച്ചു. അബ്ദുൾ കലാമി
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ക്ഷേത്ര നിർമ്മാണത്തിന്റെ ആദ്യ സൂത്രധാരൻ ശിവേയി സാമന്തരായർ എന്നയാളായിരുന്നു . ഒരിക്കൽ, ക്ഷേത്ര നിർമ്മാണം നേരത്തെ തീരുമാനിച്ച സമയത്തിനും മുന്പ് തീർക്കണമെന്ന് രാജാവ്‌ ഉത്തരവിട്ടു . അല്ലെങ്കിൽ മരണമായിരുന്നു ശിക്ഷ . എന്നാൽ തന്നെകൊണ്ട് അതിനു കഴിയില്ല എന്ന് സാമന്തരായർ രാജാവിനെ അറിയിച്ചതു പ്രകാരം രാജാവ്‌ ബിസു മഹാറാണയെ ക്ഷേത്ര നിർമ്മാണ ചുമതല  ഏൽപ്പിക്കുകയായിരുന്നു . തന്റെ  മകനായിരുന്ന ധർമപാദരുടെ സഹായത്തോടെ

[2] താജ് മഹൽ പണിതീർത്തിരിക്കുന്നത് ചുവരുകളുടെ നഗരം എന്നറിയപ്പെടുന്ന  ആഗ്ര നഗരത്തിന്റെ തെക്ക് ഭാഗത്ത് യമുന നദിയുടെ തീരത്താണ്. മഹാരാജ ജയ് സിംഗിൽ  നിന്നും വാങ്ങിയ ഭൂമിയായിരുന്നു ഇത്. പകരമായി ഷാജഹാൻ മഹാരാജ ജയ് സിങിന് ഒരു കൊട്ടാരം നൽകി എന്നാണ് പറയപ്പെടുന്നത്. മൂന്ന് ഏക്കറോളം വരുന്ന ഭൂമി ആദ്യം നിരപ്പാക്കി എടുക്കുകയും പിന്നീട് യമുന നദിയുടെ നിരപ്പിൽ നി

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[27]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[28])

QUESTION    : രാഷ്ട്രകൂടങ്ങളെ പുറത്താക്കി, കല്യാണിയിലെ ചാലൂക്യന്മാർ ഡെക്കാനിൽ തങ്ങളുടെ ഭരണം സ്ഥാപിച്ചത് ഏത് വര്ഷം ?
GOLD ANSWER : ['973-ൽ']
MODEL ANSWER: കീഴടക്കിവെച്ചു കലചൂരി
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] രാഷ്ട്രകൂടരെ പുറന്തള്ളിക്കൊണ്ട് 973-ൽ കല്യാണിയിലെ ചാലൂക്യന്മാർ ഡെക്കാണിൽ ആധിപത്യം സ്ഥാപിച്ചു. ഇരുനൂറോളം വർഷക്കാലം ഇവർ ഡെക്കാണിൽ ഭരണം തുടർന്നു. ഈ വിഭാഗത്തിലെ ഏറ്റവും ശക്തനായ രാജാവ് വിക്രമാദിത്യൻ ആറാമന്റെ (1076-1126) ഭരണകാലത്ത് ഡെക്കാണിൽ സമാധാനം നിലനിൽക്കുകയും കലയും സാഹിത്യവും അഭിവൃദ്ധി പ്രാപിക്കുകയുമുണ്ടായി. ഇദ്ദേഹത്തിന്റെ കാലശേഷം ചാലൂക്യ ശക്തി ക്രമേണ ക്ഷയിക്കുവാൻ തുടങ്ങി. ഒടുവിൽ (സു. 1190) പല സ്വത

[2] സാതവാഹനന്മാരിലെ അവസാനത്തെ രാജാവ് യജ്ഞ ശ്രീ ശാതകർണ്ണി ആയിരുന്നു. എ. ഡി. 170-മാണ്ടിൽ ഭരിച്ച അദ്ദേഹം 29 കൊല്ലത്തെ ഭരണത്തിനിടയിൽ ശകൻമാർ പിടിച്ചടക്കിയ രാജ്യഭാഗങ്ങളെല്ലാം വീണ്ടെടുത്തു. എഡി 220ൽ സതവാഹന്മാർ ക്ഷയിച്ചപ്പോൾ ഇക്ഷ്വാകു രാജവംശം, ചോള രാജവംശം, പല്ലവ രാജവംശം, ആനന്ദഗോത്രികർ, കിഴക്കൻ ചാലൂക്യന്മാർ തുടങ്ങി പല രാജവംശങ്ങൾ തെലുങ്കുദേശം ഭരിച്ചു. കടപ്പ പ്രദേശത്തുള്ള 

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[28]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

In [ ]:
show_case_ml(sample_questions_ml[29])

QUESTION    : " ഹിന്ദു "എന്ന പേര് ഇന്ത്യൻ ഉപഭൂഖണ്ഡത്തിലെ ഏത് നദിയിൽ നിന്ന് ഉത്ഭവിച്ചതാണ് ?
GOLD ANSWER : ['സിന്ധുനദിയുടെ']
MODEL ANSWER: ��ിന്ദുമതം". 

[4] ഹിന്ദുമത
Gold in retrieved passages: True

RETRIEVED PASSAGES:
[1] ഇന്ത്യൻ ഉപഭൂഖണ്ഡത്തിലെ  സിന്ധുനദിയുടെ (Indus River) പേരിൽ നിന്നാണ് ‘ഹിന്ദു’ എന്ന വാക്ക് ഉത്ഭവിച്ചതെന്ന് കരുതിപ്പോരുന്നു. ഇക്കാര്യം ആദ്യമായി  പ്രതിപാദിച്ചിരിക്കുന്നത് ഋഗ്വേദത്തിലാണ്. ഋഗ്വേദത്തിൽ ഇന്തോ-ആര്യ വംശജർ താമസിക്കുന്നിടം ‘’സപ്തസിന്ധു’’ (ഏഴ് നദികളുടെ നാട്) എന്ന് വിശേഷിപ്പിക്കപ്പെട്ടിട്ടുണ്ട്. അറബികൾ സിന്ധു നദിക്ക് അപ്പുറം നിവസിക്കുന്നവർ എന്ന അർത്ഥത്തിൽ ഉപയോഗിക്കുന്ന അൽ- ഹിന്ദ് ‘’’al-Hind’’’ എ

[2] വിതസ്താ, ചന്ദ്രഭാഗാ, ഇരാവതീ, വിപാശാ, ശതദ്രുഃ എന്നിവയാണ് പേരിനു കാരണമായ അഞ്ചുനദികൾ. നദികളോടനുബന്ധിച്ച് പുരാണകഥകളുമുണ്ട്. ഉദാഹരണത്തിന് ഒരു ചാൺ (വിതസ്തി) വീതിയുള്ള പിളർപ്പിലൂടെ പുറത്തേക്കു ചാടുന്ന സ്രോതസ്സാണത്രെ വിതസ്ത. ഈ പേര് ഝലം എന്നായിത്തീർത്തനെങ്ങനെയെന്നതിനെക്കുറിച്ച് വ്യക്തമായ സൂചനകളില്ല. ഹിമക്കട്ടകളുരുകിയുണ്ടാകുന്ന പ്രവാഹമാകയാൽ   ജലം, ഹിമം എന്ന പദങ്ങളിൽ നിന്ന് ഉരുത്

In [ ]:
labels_ml.append({
    "question_id": sample_questions_ml[29]["question_id"],
    "faithfulness": "not_supported",
    "error_type": "generator_ignored_context"
})

## 5. Check label count & summary

In [ ]:
print("Labels collected so far:", len(labels_ml))
print("Expected: 30")

assert len(labels_ml) == 30, f"Expected 30 labels, got {len(labels_ml)}"

Labels collected so far: 30
Expected: 30


In [ ]:
from collections import Counter

faithfulness_counts_ml = Counter(
    label["faithfulness"] for label in labels_ml
)

total = len(labels_ml)

print("Faithfulness breakdown:")
for label, count in faithfulness_counts_ml.items():
    print(f"  {label}: {count}/{total} ({count/total*100:.1f}%)")

Faithfulness breakdown:
  not_supported: 30/30 (100.0%)


In [ ]:
error_counts_ml = Counter(
    label["error_type"]
    for label in labels_ml
    if label["error_type"] is not None
)

print("Error type breakdown:")
for error, count in error_counts_ml.items():
    print(f"  {error}: {count}")

Error type breakdown:
  generator_ignored_context: 13
  retrieval_missed: 17


## 6. Save & push

In [ ]:
with open("results/faithfulness_labels_ml.json", "w", encoding="utf-8") as f:
    json.dump(labels_ml, f, indent=2, ensure_ascii=False)

print("Saved: results/faithfulness_labels_ml.json")

Saved: results/faithfulness_labels_ml.json


In [ ]:
!git config --global user.email "nehnamehranmk17@gmail.com"
!git config --global user.name "nehnamehrankmk638-dev"

!git add results/faithfulness_labels_ml.json
!git commit -m "ml phase1: faithfulness labels (30 manual)"
!git push

[nehna 6734a42] ml phase1: faithfulness labels (30 manual)
 1 file changed, 152 insertions(+)
 create mode 100644 results/faithfulness_labels_ml.json
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 621 bytes | 621.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git
   d55d66e..6734a42  nehna -> nehna
